# Análisis exploratorio — Mercado de motos en Cuba

Este notebook usa el dataset consolidado `data_clean/anuncios_limpios.json`.

Categorías oficiales:

1. Motos de combustión.
2. Motos eléctricas.
3. Bicicletas eléctricas.
4. Patinetes.


In [ ]:
import json
from pathlib import Path

import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = REPO / "data_clean" / "anuncios_limpios.json"

payload = json.loads(DATA.read_text(encoding="utf-8"))
df = pd.DataFrame(payload["registros"])
print(f"Registros cargados: {len(df):,}")
df[["id", "titulo", "categoria_nombre", "precio_usd", "vistas"]].head()

## Composición de la oferta


In [ ]:
composicion = (
    df["categoria_nombre"]
    .value_counts()
    .rename_axis("Categoría")
    .reset_index(name="Anuncios")
)
composicion["Porcentaje"] = (composicion["Anuncios"] / len(df) * 100).round(1)
composicion

## Precios en USD


In [ ]:
usd = df[(df["moneda"] == "USD") & df["precio_usd"].notna() & df["precio_valido"]]
precios = (
    usd.groupby("categoria_nombre")["precio_usd"]
    .agg(Anuncios="count", P25=lambda s: s.quantile(0.25), Mediana="median", P75=lambda s: s.quantile(0.75))
    .round(1)
    .reset_index()
)
precios

## Marcas con más anuncios


In [ ]:
marcas = (
    df.loc[df["marca_mostrar"].ne("SIN_MARCA"), "marca_mostrar"]
    .value_counts()
    .head(10)
    .rename_axis("Marca")
    .reset_index(name="Anuncios")
)
marcas

## Lectura rápida

- El histórico contiene 6.101 anuncios depurados.
- Las bicicletas eléctricas son la categoría más numerosa.
- La combustión concentra el precio mediano más alto.
- Los datos describen oferta publicada y vistas; no representan ventas cerradas.
